# Introduction

This notebook demonstrates the 2DIRECT sample approach used in the MatPES dataset development (http://arxiv.org/abs/2503.04070) that ensures comprehensive coverage in both structural and atomic feature space. 

In [1]:
from __future__ import annotations

import warnings

import numpy as np
import plotly.graph_objects as go
from pymatgen.core import Element, Structure
from pymatgen.io.lammps.outputs import parse_lammps_dumps

from maml.describers import MatGLSite, MatGLStructure

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
# Initialize Atom and Structure-level M3GNet featurizers
featurizer_atom = MatGLSite(output_layers=["gc_1"])
featurizer_structure = MatGLStructure()

/Users/shyue/repos/maml/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sampling structures from two 100 ps MD trajectories of Li₃PS₄

**Goal:** Cover atomic and structural feature space with the smallest possible structures.

- 1,001 snapshots per trajectory
- Initial structures: Li₃PS₄ (unit cell) and Li₂₄P₈S₃₂ (supercell)

## 1.1 Load structures

In [3]:
def read_lammps_dumped_structures(filename: str, filtered_elements=("Li", "P", "S")):
    """
    Read the dumped structures from a file and filter them by given elements.

    Args:
        filename: Path to the LAMMPS dump file containing the structures.
        filtered_elements: Tuple of element symbols to include. Defaults to ("Li", "P", "S").

    Returns:
        list: A list of pymatgen Structure objects.
    """
    structures = []
    filtered_elements = [str(e) for e in sorted(Element(el) for el in filtered_elements)]
    for i in parse_lammps_dumps(filename):
        L = i.box.to_lattice()
        atom_position = i.data.sort_values(by="id")
        species = [filtered_elements[t - 1] for t in atom_position["type"]]
        x = np.array(atom_position["x"]).reshape(len(species), 1)
        y = np.array(atom_position["y"]).reshape(len(species), 1)
        z = np.array(atom_position["z"]).reshape(len(species), 1)
        coordinate = np.concatenate((x, y, z), axis=1)
        structure = Structure(L, species, coordinate, coords_are_cartesian=True)
        structures.append(structure)
    return structures

In [4]:
md_supercells = read_lammps_dumped_structures("dump.100ps.mp-2646995_64_Li24P8S32")
md_unitcells = read_lammps_dumped_structures("dump.100ps.mp-2646995_8_Li3P1S4")
structures = md_supercells + md_unitcells
print(f"Structures: {len(md_supercells)} supercells, {len(md_unitcells)} unit cells")
print(f"Supercell sizes: { {len(s) for s in md_supercells} }")
print(f"Unit cell sizes: { {len(s) for s in md_unitcells} }")

Structures: 1001 supercells, 1001 unit cells
Supercell sizes: {64}
Unit cell sizes: {8}


## 1.2 Structural and atomic features

In [5]:
atom_features_supercells = [
    featurizer_atom.transform_one(s).to_numpy() for s in md_supercells
]
atom_features_unitcells = [
    featurizer_atom.transform_one(s).to_numpy() for s in md_unitcells
]
struct_features_supercells = [
    featurizer_structure.transform_one(s) for s in md_supercells
]
struct_features_unitcells = [
    featurizer_structure.transform_one(s) for s in md_unitcells
]

at_ft_supercell_arr = np.array(atom_features_supercells)
at_ft_unitcell_arr = np.array(atom_features_unitcells)
st_ft_supercell_arr = np.array(struct_features_supercells)
st_ft_unitcell_arr = np.array(struct_features_unitcells)

print(f"Shape of supercell atom features: {at_ft_supercell_arr.shape}")
print(f"Shape of supercell structure features: {st_ft_supercell_arr.shape}")
print(f"Shape of unit cell atom features: {at_ft_unitcell_arr.shape}")
print(f"Shape of unit cell structure features: {st_ft_unitcell_arr.shape}")

Shape of supercell atom features: (1001, 64, 64)
Shape of supercell structure features: (1001, 128)
Shape of unit cell atom features: (1001, 8, 64)
Shape of unit cell structure features: (1001, 128)


## 1.3 Structural and atomic feature space: supercells vs unit cells

The inverse trend in coverage by unit cells and supercells in the two feature spaces is discussed in the MatPES manuscript.

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=st_ft_unitcell_arr[:, 0], y=st_ft_unitcell_arr[:, 1],
    mode="markers",
    name="Li3PS4 unit cell",
    marker=dict(size=6, opacity=0.6, color="#2E86AB", line=dict(width=0)),
))
fig.add_trace(go.Scatter(
    x=st_ft_supercell_arr[:, 0], y=st_ft_supercell_arr[:, 1],
    mode="markers",
    name="Li24P8S32 supercell",
    marker=dict(size=6, opacity=0.7, color="#E94F37", line=dict(width=0)),
))
fig.update_layout(
    title="M3GNet structural feature space",
    xaxis_title="1st dimension",
    yaxis_title="2nd dimension",
    template="plotly_white",
    height=500,
    width=500,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.show()

In [7]:
atom_features_unitcells_flatten = np.array([af for afs in atom_features_unitcells for af in afs])
atom_features_supercells_flatten = np.array([af for afs in atom_features_supercells for af in afs])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=atom_features_supercells_flatten[:, 0],
    y=atom_features_supercells_flatten[:, 1],
    mode="markers",
    name=f"Li₂₄P₈S₃₂ ({len(atom_features_supercells_flatten):,} atomic envs)",
    marker=dict(size=4, opacity=0.7, color="#2E86AB", line=dict(width=0)),
))
fig.add_trace(go.Scatter(
    x=atom_features_unitcells_flatten[:, 0],
    y=atom_features_unitcells_flatten[:, 1],
    mode="markers",
    name=f"Li₃PS₄ ({len(atom_features_unitcells_flatten):,} atomic envs)",
    marker=dict(size=4, opacity=0.5, color="#E94F37", line=dict(width=0)),
))
fig.update_layout(
    title="M3GNet atomic feature space",
    xaxis_title="PCA1",
    yaxis_title="PCA2",
    template="plotly_white",
    height=600,
    width=600,
    legend=dict(yanchor="middle", y=0.5, xanchor="left", x=0.01),
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.show()

## 1.4 2DIRECT sampling — example

2DIRECT samples structures with representative atomic environments, preferring the smallest structures to minimize cost and using larger cells only when needed. See the manuscript for details.

**Note:** The original DIRECT (structural-feature space only) is sufficient when unit cells and supercells are not mixed.

In this particular example, supercells and unit cells are mixed, and we show how 2DIRECT compares with the original DIRECT sampling. We can see that:

1. 2DIRECT better samples atomic feature space than the original DIRECT.
2. 2DIRECT sampled structures have a lower number of sites than the average of all structures, as large supercells are sampled only when necessary.

### Step 1: Partition structures by structural feature space

Partition the 2,002 structures into 20 groups using their location in structural feature space.

In [8]:
from maml.sampling.direct import BirchClustering, DIRECTSampler, SelectKFromClusters

DIRECT_partitioner = DIRECTSampler(
    structure_encoder=None,
    clustering=BirchClustering(n=20, threshold_init=0.05),
    select_k_from_clusters=None
)

In [9]:
%%time
all_struct_features = list(struct_features_supercells) + list(struct_features_unitcells)
DIRECT_partition = DIRECT_partitioner.fit_transform(all_struct_features)

INFO:maml.sampling.pca:Selected first 13 PCs, explaining 93.31% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=20 gives 20 clusters.


CPU times: user 584 ms, sys: 1.68 s, total: 2.26 s
Wall time: 214 ms


### Step 2: DIRECT sampling within each cluster

From each cluster, sample by atomic features and structure size (preferring smaller structures).

In [10]:
%%time
all_atom_features = atom_features_supercells + atom_features_unitcells
sampled_structure_ids = []

for label in set(DIRECT_partition["labels"]):
    structure_ids = np.where(DIRECT_partition["labels"] == label)[0]
    # Flatten atom features and site counts for structures in this cluster
    atom_features = [
        af
        for i, afs in enumerate(all_atom_features)
        if i in structure_ids
        for af in afs
    ]
    nsites = [
        len(s)
        for i, s in enumerate(structures)
        if i in structure_ids
        for _ in range(len(s))
    ]
    structure_ids_atomf = [
        i
        for i, s in enumerate(structures)
        if i in structure_ids
        for _ in range(len(s))
    ]

    direct_sampler = DIRECTSampler(
        None,
        clustering=BirchClustering(n=40, threshold_init=0.05),
        select_k_from_clusters=SelectKFromClusters(
            selection_criteria="smallest", n_sites=nsites
        ),
    )
    DIRECT_sampling = direct_sampler.fit_transform(np.array(atom_features))
    struct_ids = {
        structure_ids_atomf[i] for i in DIRECT_sampling["selected_indexes"]
    }
    sampled_structure_ids.extend(struct_ids)

sampled_structure_ids = list(set(sampled_structure_ids))
print(f"Selected in total {len(sampled_structure_ids)} smallest possible structures with 2DIRECT")

INFO:maml.sampling.pca:Selected first 3 PCs, explaining 95.51% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=40 gives 40 clusters.
INFO:maml.sampling.stratified_sampling:Finally selected 40 configurations.
INFO:maml.sampling.pca:Selected first 3 PCs, explaining 95.36% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=40 gives 40 clusters.
INFO:maml.sampling.stratified_sampling:Finally selected 40 configurations.
INFO:maml.sampling.pca:Selected first 3 PCs, explaining 95.71% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=40 gives 40 clusters.
INFO:maml.sampling.stratified_sampling:Finally selected 40 configurations.
INFO:maml.sampling.pca:Selected first 3 PCs, explaining 95.30% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=40 gives 40 clusters.
INFO:maml.sampling.stratified_sampling:Finally selected 40 configurations.
INFO:maml.sampli

Selected in total 420 smallest possible structures with 2DIRECT
CPU times: user 6.83 s, sys: 21 s, total: 27.8 s
Wall time: 2.52 s


In [11]:
def plot_feature_coverage(
    selected_indexes,
    method="2DIRECT",
    all_features=None,
    all_features_atoms=None
):
    """
    Plots the feature coverage based on the selected indexes.

    Parameters:
    selected_indexes (list or array): Indices of the selected features to plot.
    method (str, optional): The method used for feature selection (default is "2DIRECT").
    all_features (numpy array, optional): All available features to plot (default is None).
    all_features_atoms (list or array, optional): Atom-specific features to plot.

    Returns:
    None: The function generates a plot.
    """
    if all_features is None:
        all_features = np.array([struct_features_supercells + struct_features_unitcells][0])

    if all_features_atoms is None:
        all_features_atoms = atom_features_supercells + atom_features_unitcells

    selected_features = all_features[selected_indexes]
    ave_all = round(np.mean([len(s) for s in structures]))
    ave_selected = round(np.mean([len(s) for i, s in enumerate(structures) if i in selected_indexes]))
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=all_features[:, 0], y=all_features[:, 1],
        mode="markers",
        name=f"All {len(all_features):,} structures (n_site,avg={ave_all})",
        marker=dict(size=5, opacity=0.5, color="#7f7f7f", line=dict(width=0)),
    ))
    fig.add_trace(go.Scatter(
        x=selected_features[:, 0], y=selected_features[:, 1],
        mode="markers",
        name=f"{method} sampled {len(selected_features):,} (n_site,avg={ave_selected})",
        marker=dict(size=5, opacity=0.7, color="#2E86AB", line=dict(width=0)),
    ))
    fig.update_layout(
        title="M3GNet structural feature — coverage",
        xaxis_title="1st dimension",
        yaxis_title="2nd dimension",
        template="plotly_white",
        height=500,
        width=500,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        margin=dict(t=60, b=50, l=60, r=40),
    )
    fig.show()

    all_features_atoms_flatten = np.array([af for afs in all_features_atoms for af in afs])
    atom_features_atom_selected_flatten = np.array([
        af for i, afs in enumerate(all_features_atoms) if i in selected_indexes for af in afs
    ])
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=all_features_atoms_flatten[:, 0], y=all_features_atoms_flatten[:, 1],
        mode="markers",
        name=f"All {len(all_features_atoms_flatten):,} atomic environments",
        marker=dict(size=3, opacity=0.5, color="#7f7f7f", line=dict(width=0)),
    ))
    fig2.add_trace(go.Scatter(
        x=atom_features_atom_selected_flatten[:, 0],
        y=atom_features_atom_selected_flatten[:, 1],
        mode="markers",
        name=f"{method} selected {len(atom_features_atom_selected_flatten):,}",
        marker=dict(size=3, opacity=0.7, color="#E94F37", line=dict(width=0)),
    ))
    fig2.update_layout(
        title="M3GNet atomic feature — coverage",
        xaxis_title="1st dimension",
        yaxis_title="2nd dimension",
        template="plotly_white",
        height=500,
        width=500,
        legend=dict(yanchor="middle", y=0.5, xanchor="left", x=0.01),
        margin=dict(t=60, b=50, l=60, r=40),
    )
    fig2.show()

In [12]:
%%time
plot_feature_coverage(sampled_structure_ids)

CPU times: user 133 ms, sys: 660 ms, total: 793 ms
Wall time: 70.2 ms


In [13]:
%%time
direct_original = DIRECTSampler(None, clustering=BirchClustering(n=392, threshold_init=0.05))
DIRECT_original = direct_original.fit_transform(struct_features_supercells+struct_features_unitcells)
sampled_structure_ids_original = DIRECT_original["selected_indexes"]

INFO:maml.sampling.pca:Selected first 13 PCs, explaining 93.31% variance
INFO:maml.sampling.clustering:BirchClustering with threshold_init=0.05 and n=392 gives 392 clusters.
INFO:maml.sampling.stratified_sampling:Finally selected 392 configurations.


CPU times: user 1.13 s, sys: 2.15 s, total: 3.28 s
Wall time: 301 ms


In [14]:
%%time
plot_feature_coverage(sampled_structure_ids_original, method="Original DIRECT")

CPU times: user 161 ms, sys: 509 ms, total: 670 ms
Wall time: 63.7 ms


# Exporting the Structures

The next step is to generate input files for the sampled structures. Obviously this will be highly dependent on the input set you are using. Here we will demonstrate using the MatPES input set that is compatible with MatPES based foundation potentials.

In [15]:
from pymatgen.io.vasp.sets import MatPESStaticSet

for i in sampled_structure_ids:
    vis = MatPESStaticSet(structures[i])
    # This next line is deliberately commented out to avoid outputting a lot of files. For actual use, you should
    # uncomment it.
    # vis.write_input(f"sampled_structure_{i}")